In [ ]:
!pip install simpletransformers

In [ ]:
# accessing google drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/

In [ ]:
from simpletransformers.classification import ClassificationModel, ClassificationArgs

# Optional model configuration
model_args = ClassificationArgs(num_train_epochs=1,overwrite_output_dir=True,train_batch_size=1,output_dir='/content/drive/MyDrive/op_dir/')

# Create a ClassificationModel
model = ClassificationModel(
    'bert',
    'bert-base-cased',
    use_cuda=False,
    num_labels=10,
    args=model_args
)

In [ ]:
import pandas as pd

# Preparing the dataset
dataset = pd.read_csv('short_stories_minus_dystopian.csv', encoding = 'latin-1')
dyst_dataset = pd.read_csv('stories_dystopian.csv', encoding = 'latin-1')
dataset = pd.concat([dataset[:300], dyst_dataset, dataset[300:]])
dataset = dataset[['Text', 'Genre_Code']]

In [ ]:
# randomly shuffling the dataset for better results
dataset = dataset.sample(frac = 1)
train_df = dataset[:700]
eval_df = dataset[700:850]
test_df = dataset[850:]

In [ ]:
# Train the model
model.train_model(train_df)

In [ ]:
# Evaluate the model
# result, model_outputs, wrong_predictions = model.eval_model(eval_df)
model.eval_model(eval_df)

In [ ]:
# Make predictions with the model
i = 1
count = 0
output = []
for test_story in test_df['Text']:
  print('Entry Number = ', i)
  output.append(model.predict(test_story)[0][0])
  i += 1

In [ ]:
result = list(test_df['Genre_Code'])

In [ ]:
count = 0
for i in range(len(test_df['Text'])):
  if output[i] == result[i]:
    count += 1

In [ ]:
count/len(test_df['Text']) * 100

In [ ]:
test_story = '''
Leiningen is an industrious coffee plantation owner in Brazil. He has built his farm up through hard work and ingenuity. When warned of an approaching cataclysm -- a large swarm of ants eating their way through the jungle as an unstoppable force consuming everything in their path -- Leiningen refuses to yield like his neighbors:

The Brazilian official threw up lean and lanky arms and clawed the air with wildly distended fingers. "Leiningen!" he shouted. "You're insane! They're not creatures you can fight--they're an elemental--an 'act of God!' Ten miles long, two miles wide--ants, nothing but ants! And every single one of them a fiend from hell; before you can spit three times they'll eat a full-grown buffalo to the bones. I tell you if you don't clear out at once there'll be nothing left of you but a skeleton picked as clean as your own plantation."

The plantation workers are frightened, Leiningen offers them full pay and allows them to leave. But the boss is such a force of nature himself that some workers stay behind to implement Leiningen's formidable battle plans to defeat the ants.

Using a series of canals, ruses to draw the ants off course, and petroleum fires, the people and the ants engage in a days-long battle. The canals prove effective at first, but soon, the ants learn to cross the canals. Some ants willfully jump into the canals and drown, but their floating bodies are pulled into rafted bridges and that the other ants use to cross. The battle rages on over days and the humans are trapped in a concrete ditch ringed with fire.

All seems lost, then Leiningen realizes that he could breach a dam, two miles away. While that will flood his plantation and destroy the crops, the people will live. Despite the ferocity of the ants, Leiningen determines to wrap himself in petroleum rags and run two miles through the ants to get to the dam and breach it. On his way back he stumbles. He recalls witnessing the ants once pick a downed stag clean to the "white bones" and he resolves not to die that way. Leiningen struggles to his feet and returns to safety. He is badly hurt but alive. At the end of the story he is recovering and states, "I told you I would come back, even if I am a bit streamlined."
'''
model.predict([test_story])

In [ ]:
import pickle

# save the model to disk
pickle.dump(model, open('/content/drive/MyDrive/model_saved_stories', 'wb'))